#### **OptaLive: Data Exploration Notebook**
**Objective:** Inspect the raw StatsBomb Open Data (EPL 2015/2016 season) to understand match metadata, team structures, lineups, and live event schemas before building the validation pipeline.

#### **Imports & Environment Setup**

In [1]:
import os
import json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")
print("Current working directory:", os.getcwd())

Libraries imported successfully!
Current working directory: C:\Users\tohiba\Desktop\optalive-qa-dashboard\notebooks


#### **1. Match Metadata Inspection**
We load `matches_epl_2015_2016.json` to see available matches, home/away teams, stadiums, and match dates.

#### **Load Matches DataFrame**

In [3]:
matches_path = os.path.join("..", "data", "raw", "matches_epl_2015_2016.json")

with open(matches_path, "r", encoding="utf-8") as f:
    matches_data = json.load(f)

# Use json_normalize to flatten nested fields (home_team, away_team, stadium, etc.)
df_matches = pd.json_normalize(matches_data)
print(f"Total matches loaded: {len(df_matches)}")

# Dynamically find the correct column names for home team, away team, and stadium
home_col = 'home_team.home_team_name' if 'home_team.home_team_name' in df_matches.columns else 'home_team.name'
away_col = 'away_team.away_team_name' if 'away_team.away_team_name' in df_matches.columns else 'away_team.name'
stadium_col = 'stadium.name' if 'stadium.name' in df_matches.columns else [c for c in df_matches.columns if 'stadium' in c][0]

# Inspect key columns safely
cols_to_show = ['match_id', 'match_date', home_col, away_col, stadium_col]
print("\nSample match records:")
print(df_matches[cols_to_show].head(5))

Total matches loaded: 380

Sample match records:
   match_id  match_date home_team.home_team_name away_team.away_team_name  \
0   3754217  2015-09-19                  Chelsea                  Arsenal   
1   3754117  2015-12-13              Aston Villa                  Arsenal   
2   3754296  2015-12-21                  Arsenal          Manchester City   
3   3753983  2015-10-31             Swansea City                  Arsenal   
4   3754160  2015-12-05                  Arsenal               Sunderland   

          stadium.name  
0      Stamford Bridge  
1           Villa Park  
2     Emirates Stadium  
3  Swansea.com Stadium  
4     Emirates Stadium  


#### **2. Lineup & Team Sheet Inspection**
Let's examine the lineup structure for a single match (e.g., Match ID `3754217`). We need to ensure every starting XI has 11 players, shirt numbers, and designated positions.

#### **Inspect Sample Lineup JSON**

In [4]:
sample_match_id = 3754217
lineup_path = os.path.join("..", "data", "raw", "lineups", f"{sample_match_id}.json")

with open(lineup_path, "r", encoding="utf-8") as f:
    lineup_data = json.load(f)

print(f"Teams in lineup file: {[team['team_name'] for team in lineup_data]}")

# Inspect first team's starting lineup record structure
first_team = lineup_data[0]
print(f"\nTeam: {first_team['team_name']} | Total squad players: {len(first_team['lineup'])}")

# Show sample player record
sample_player = first_team['lineup'][0]
print("\nSample player dictionary structure:")
print(json.dumps(sample_player, indent=2))

Teams in lineup file: ['Chelsea', 'Arsenal']

Team: Chelsea | Total squad players: 18

Sample player dictionary structure:
{
  "player_id": 3339,
  "player_name": "Asmir Begovi\u0107",
  "player_nickname": null,
  "jersey_number": 1,
  "country": {
    "id": 28,
    "name": "Bosnia and Herzegovina"
  },
  "cards": [],
  "positions": [
    {
      "position_id": 1,
      "position": "Goalkeeper",
      "from": "00:00",
      "to": null,
      "from_period": 1,
      "to_period": null,
      "start_reason": "Starting XI",
      "end_reason": "Final Whistle"
    }
  ]
}


#### **3. Event-Level Stream Inspection**
This is the core of OptaLive. We load the raw event feed to find goals, fouls with cards, and substitutions.

#### **Load & Flatten Event Stream**

In [5]:
events_path = os.path.join("..", "data", "raw", "events", f"{sample_match_id}.json")

with open(events_path, "r", encoding="utf-8") as f:
    events_data = json.load(f)

df_events = pd.json_normalize(events_data)
print(f"Total raw events captured for match {sample_match_id}: {len(df_events)}")

# Check unique event types in this match
print("\nTop Event Types:")
print(df_events['type.name'].value_counts().head(10))

Total raw events captured for match 3754217: 3732

Top Event Types:
type.name
Pass             1046
Ball Receipt*     963
Carry             814
Pressure          320
Ball Recovery     106
Duel               78
Dribble            52
Clearance          42
Block              41
Goal Keeper        34
Name: count, dtype: int64


#### **Filter Key OptaLive Events: Shots, Cards, Substitutions**

In [6]:
# Filter for Shots (potential goals), Fouls (cards), and Substitutions
key_types = ["Shot", "Foul Committed", "Substitution", "Starting XI"]
df_filtered = df_events[df_events['type.name'].isin(key_types)].copy()

print(f"Total filtered key events: {len(df_filtered)}")

# Inspect specific sub-properties like shot outcome or card types
if 'shot.outcome.name' in df_filtered.columns:
    print("\nShot outcomes:")
    print(df_filtered['shot.outcome.name'].value_counts(dropna=False))

if 'foul_committed.card.name' in df_filtered.columns:
    print("\nCard types recorded:")
    print(df_filtered['foul_committed.card.name'].value_counts(dropna=False))

Total filtered key events: 73

Shot outcomes:
shot.outcome.name
NaN        40
Blocked    13
Off T       9
Saved       7
Wayward     3
Goal        1
Name: count, dtype: int64

Card types recorded:
foul_committed.card.name
NaN              68
Yellow Card       4
Second Yellow     1
Name: count, dtype: int64


#### **Summary of Findings & In-Ground Data Insights**

- **Matches Metadata:** Successfully loaded all 380 Premier League 2015/2016 fixtures, providing reliable team IDs, dates, and venue information.
- **Lineups & Team Sheets:** Confirmed complete squad structures containing official player names, jersey numbers, nationalities, and tactical starting positions.
- **Event Streams:** Extracted granular match events from nested JSON structures, filtering down 3,700+ raw actions into 73 critical key events (such as goals via `shot.outcome.name == 'Goal'` and disciplinary actions via `foul_committed.card.name`).

**Next Step:** Transition to **`02_data_cleaning.ipynb`** to formalize these extraction rules into our reusable production modules (`src/data_loader.py` and `src/preprocessing.py`).